# Milestone 4 Ka Primary Goal
Milestone 1-3 mein hum text ko encode kar rahe the, search kar rahe the, ya generic zero-shot pipelines chalaye the.

Milestone 4 mein hum Model ko actual Multiple-Choice Classification seekha rahe hain aur wo bhi LoRA (Parameter-Efficient Fine-Tuning) ka use karke.

Is Milestone ke main 4 Core Pillars hain:

[1. Data Formatting] ──► [2. AutoModelForMultipleChoice] ──► [3. PEFT / LoRA] ──► [4. Trainer & Inference]
(5 Options to 5 Pairs)     (Logits & 3D Tensors)           (Freeze Base + Adapters)     (Fine-tune & Softmax)

# Question 1: Label Encoding

In [1]:
# ==========================================================
# Question 1
# Encode categorical answer labels into numeric labels
# ==========================================================

import pandas as pd

train = pd.read_csv("../data/train.csv")

label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train["label"] = train["answer"].map(label_map)

original_answer = train.loc[150, "answer"]
encoded_label = train.loc[150, "label"]

print("Original Answer :", original_answer)
print("Encoded Label   :", encoded_label)

Original Answer : C
Encoded Label   : 2


# Question 2: Prompt-Option Formatting

For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?

### Concept

In multiple-choice classification, each answer option is paired separately with the original question prompt.

For example, a question with five options is transformed into five separate prompt-option pairs:

- Prompt + Option A
- Prompt + Option B
- Prompt + Option C
- Prompt + Option D
- Prompt + Option E

The `[SEP]` separator is used here to explicitly separate the question from the candidate answer option.

In [2]:
# ==========================================================
# Question 2
# Format Prompt with Option B and calculate character length
# ==========================================================

# Select row index 0
row_0 = train.iloc[0]

# Extract the prompt and Option B
prompt = str(row_0["prompt"])
option_b = str(row_0["B"])

# Create the formatted input using the exact required format
formatted_input = prompt + " [SEP] " + option_b

# Calculate exact character length
character_length = len(formatted_input)

print("Formatted Input:")
print(formatted_input)
print()
print("Character Length :", character_length)

Formatted Input:
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.

Character Length : 407


The exact character length of the formatted **Prompt + Option B** input is **407 characters**.

In a multiple-choice classification task, the same process is repeated for all five options, creating five separate prompt-option pairs for each question.

# Question 3: Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:

padding = "max_length", truncation = True, max_length = 128, return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:

[1, 5, 128]

What is the value of the second dimension?

Then reshape the tokenized inputs into the format expected by a multiple-choice model:

`[batch_size, num_choices, sequence_length]`


In [3]:
# ==========================================================
# Question 3
# Tokenize five choices for a single MCQ
# ==========================================================

import torch
from transformers import AutoTokenizer

# Load the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row_0 = train.iloc[0]

prompt = str(row_0["prompt"])

# Define the five answer choices
option_letters = ["A", "B", "C", "D", "E"]

# Create five formatted prompt-option inputs
formatted_inputs = [
    prompt + " [SEP] " + str(row_0[option])
    for option in option_letters
]

# Tokenize all five choices
encoded = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Current shape: [5, 128]
print("Shape before reshaping :", encoded["input_ids"].shape)

# Add batch dimension
input_ids = encoded["input_ids"].unsqueeze(0)

# Final shape: [1, 5, 128]
print("Final input_ids shape  :", input_ids.shape)

# Extract the second dimension
second_dimension = input_ids.shape[1]

print("Second Dimension       :", second_dimension)

e:\Projects\GenAi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Shape before reshaping : torch.Size([5, 128])
Final input_ids shape  : torch.Size([1, 5, 128])
Second Dimension       : 5


### Conclusion

The second dimension of the final `input_ids` tensor is **5**.

This dimension represents the five answer choices **A, B, C, D, and E** for a single multiple-choice question. The resulting tensor shape `[1, 5, 128]` is compatible with the input structure expected by multiple-choice classification models.

# Question 4: Batch MCQ Tokenization

## Objective

Tokenize the first **16 rows** of the training dataset as multiple-choice examples.

Each question contains **5 answer choices**, and every prompt-option pair is tokenized to a fixed sequence length of **128 tokens**.

The resulting `input_ids` tensor should have the shape:

`[16, 5, 128]`

Finally, calculate the total number of token positions present in this tensor.

In [4]:
# ==========================================================
# Question 4
# Batch Multiple-Choice Tokenization
# ==========================================================

# Select the first 16 rows
batch_df = train.iloc[:16]

batch_input_ids = []

# Process each question separately
for _, row in batch_df.iterrows():

    prompt = str(row["prompt"])

    # Create five prompt-option pairs
    formatted_inputs = [
        prompt + " [SEP] " + str(row[option])
        for option in ["A", "B", "C", "D", "E"]
    ]

    # Tokenize all five choices
    encoded = tokenizer(
        formatted_inputs,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    # Shape of each question: [5, 128]
    batch_input_ids.append(encoded["input_ids"])


# Stack all 16 questions into a single tensor
# Final shape: [16, 5, 128]
batch_input_ids = torch.stack(batch_input_ids)

# Calculate total number of token positions
total_token_positions = batch_input_ids.numel()

print("Final input_ids shape :", batch_input_ids.shape)
print("Total Token Positions :", total_token_positions)

Final input_ids shape : torch.Size([16, 5, 128])
Total Token Positions : 10240


The final multiple-choice input tensor contains exactly **10,240 token positions**.

The tensor shape `[16, 5, 128]` represents:

- **16** questions in the batch
- **5** candidate choices per question
- **128** token positions per choice

This demonstrates how multiple MCQ examples are organized into the three-dimensional tensor format expected by multiple-choice classification models.

# Question 5: Multiple-Choice Logits

## Objective

Load the pre-trained **bert-base-uncased** model using `AutoModelForMultipleChoice`.

Tokenize all five prompt-option pairs for row index **0**, pass them through the model, and inspect the shape of the output logits tensor.

The goal is to determine how many logits the model produces for one multiple-choice question containing five answer options.

In [ ]:
# ==========================================================
# Question 5
# Generate Multiple-Choice Logits using BERT
# ==========================================================

import torch
from transformers import AutoModelForMultipleChoice

# Load the pre-trained BERT multiple-choice model
model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

# Select row index 0
row_0 = train.iloc[0]

prompt = str(row_0["prompt"])

# Create five prompt-option pairs
formatted_inputs = [
    prompt + " [SEP] " + str(row_0[option])
    for option in ["A", "B", "C", "D", "E"]
]

# Tokenize the five choices
encoded = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Add the batch dimension:
# [5, 128] -> [1, 5, 128]
input_ids = encoded["input_ids"].unsqueeze(0)
attention_mask = encoded["attention_mask"].unsqueeze(0)

# Disable gradient calculation because we are only doing inference
model.eval()

with torch.no_grad():

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

# Extract logits
logits = outputs.logits

print("Logits       :", logits)
print("Logits Shape :", logits.shape)
print("Total Logits :", logits.shape[1])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2398.01it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Cons

Logits       : tensor([[-0.2262, -0.1971, -0.2006, -0.2121, -0.2160]])
Logits Shape : torch.Size([1, 5])
Total Logits : 5


### Conclusion

The multiple-choice model produces **5 logits** for one question because the question contains five candidate answer choices.

Each logit represents the model's raw score for one answer option. These logits can later be converted into probabilities using the Softmax function.

# Question 6: Supervised Loss Tensor

## Objective

Pass the tokenized five-choice input for row index **0** into the `AutoModelForMultipleChoice` model along with its correct encoded label.

The model will compute a supervised loss value in addition to the logits.

The objective is to determine the number of dimensions in the returned loss tensor.

In [6]:
# ==========================================================
# Question 6
# Compute Supervised Loss for Multiple-Choice Classification
# ==========================================================

# Get the correct encoded label for row index 0
correct_label = int(row_0["label"])

# Convert the label into a tensor with batch shape [1]
labels = torch.tensor([correct_label])

# Pass inputs and correct label through the model
model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

# Extract the scalar loss tensor
loss = outputs.loss

print("Correct Answer        :", row_0["answer"])
print("Encoded Label         :", correct_label)
print("Loss                  :", loss)
print("Loss Shape            :", loss.shape)
print("Number of Dimensions  :", loss.dim())

Correct Answer        : B
Encoded Label         : 1
Loss                  : tensor(1.5962)
Loss Shape            : torch.Size([])
Number of Dimensions  : 0


### Conclusion

The supervised loss tensor returned by the multiple-choice model has **0 dimensions**.

This is because the classification loss is reduced to a single scalar value representing the overall prediction error for the given batch.

# Question 7: LoRA Trainable Parameters

## Objective

Apply **Low-Rank Adaptation (LoRA)** to the `bert-base-uncased` multiple-choice model using the specified configuration:

- `r = 8`
- `lora_alpha = 16`
- `target_modules = ["query", "value"]`
- `lora_dropout = 0.1`
- `bias = "none"`
- `task_type = TaskType.SEQ_CLS`

Then calculate the total number of trainable parameters after applying LoRA.

In [7]:
# ==========================================================
# Question 7
# Apply LoRA and Count Trainable Parameters
# ==========================================================

from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, TaskType, get_peft_model

# Load a fresh BERT multiple-choice model
lora_model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

# Define the required LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

# Apply LoRA adapters to the model
lora_model = get_peft_model(
    lora_model,
    lora_config
)

# Count trainable and total parameters
trainable_params = sum(
    p.numel()
    for p in lora_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in lora_model.parameters()
)

trainable_percentage = (
    100 * trainable_params / total_params
)

print(f"Trainable Parameters : {trainable_params:,}")
print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Percentage : {trainable_percentage:.4f}%")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5813.05it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Cons

Trainable Parameters : 295,681
Total Parameters     : 109,778,690
Trainable Percentage : 0.2693%


### Conclusion

The number of trainable parameters after applying LoRA is **295,681**.

Only **0.2693%** of the total model parameters are trainable, demonstrating the parameter efficiency of LoRA. Instead of updating the entire pre-trained BERT model, LoRA introduces and trains a small number of adapter parameters in selected attention modules.

# Question 8: Hugging Face Dataset Preparation

## Objective

Create a Hugging Face `Dataset` from the first **100 rows** of the training dataset.

For each multiple-choice question:

- Create 5 prompt-option pairs.
- Tokenize every pair to a maximum length of 128.
- Store `input_ids` with shape `[5, 128]`.
- Store `attention_mask` with shape `[5, 128]`.
- Store the correct encoded answer as `labels`.

Finally, inspect the first dataset item and determine how many tokenized choices are stored in its `input_ids`.

In [8]:
# ==========================================================
# Question 8
# Prepare Hugging Face Dataset for Multiple-Choice Training
# ==========================================================

import torch
from datasets import Dataset

# Select the first 100 rows
train_100 = train.iloc[:100].reset_index(drop=True)

# Define all answer choices
option_letters = ["A", "B", "C", "D", "E"]

# Lists to store processed examples
all_input_ids = []
all_attention_masks = []
all_labels = []

# Process each row as one multiple-choice example
for _, row in train_100.iterrows():

    prompt = str(row["prompt"])

    # Create five prompt-option pairs
    formatted_inputs = [
        prompt + " [SEP] " + str(row[option])
        for option in option_letters
    ]

    # Tokenize all five choices
    encoded = tokenizer(
        formatted_inputs,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    # Store tokenized choices
    all_input_ids.append(encoded["input_ids"])
    all_attention_masks.append(encoded["attention_mask"])

    # Store the correct encoded answer
    all_labels.append(int(row["label"]))


# Create Hugging Face Dataset
hf_dataset = Dataset.from_dict({
    "input_ids": all_input_ids,
    "attention_mask": all_attention_masks,
    "labels": all_labels
})


# Inspect the first dataset item
first_item = hf_dataset[0]

# Convert to tensors only for checking shapes
first_input_ids = torch.tensor(first_item["input_ids"])
first_attention_mask = torch.tensor(first_item["attention_mask"])

print("Dataset Size         :", len(hf_dataset))
print("Input IDs Shape      :", first_input_ids.shape)
print("Attention Mask Shape :", first_attention_mask.shape)
print("Label                :", first_item["labels"])
print("Number of Choices    :", first_input_ids.shape[0])

Dataset Size         : 100
Input IDs Shape      : torch.Size([5, 128])
Attention Mask Shape : torch.Size([5, 128])
Label                : 1
Number of Choices    : 5


### Conclusion

The first dataset item's `input_ids` contains **5 tokenized choices**.

Each choice is represented by a sequence of **128 token positions**, resulting in the shape `[5, 128]`. This structured format can now be used by the Hugging Face Trainer for multiple-choice model fine-tuning.

# Question 9: Tiny LoRA Fine-Tuning

## Objective

Fine-tune a LoRA-enabled `bert-base-uncased` multiple-choice model on the first **32 rows** of the training dataset using the Hugging Face `Trainer`.

The specified training configuration is:

- Maximum sequence length: `64`
- Training batch size: `4`
- Gradient accumulation steps: `1`
- Maximum training steps: `4`

After training, inspect the final `global_step` reported by the Trainer.

In [9]:
# ==========================================================
# Question 9
# Tiny LoRA Fine-Tuning using Hugging Face Trainer
# ==========================================================

import torch
from datasets import Dataset
from transformers import (
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, TaskType, get_peft_model


# ----------------------------------------------------------
# Step 1: Prepare the first 32 training examples
# ----------------------------------------------------------

train_32 = train.iloc[:32].reset_index(drop=True)

option_letters = ["A", "B", "C", "D", "E"]

q9_input_ids = []
q9_attention_masks = []
q9_labels = []


for _, row in train_32.iterrows():

    prompt = str(row["prompt"])

    # Create five prompt-option pairs
    formatted_inputs = [
        prompt + " [SEP] " + str(row[option])
        for option in option_letters
    ]

    # Tokenize with max_length = 64 as required
    encoded = tokenizer(
        formatted_inputs,
        padding="max_length",
        truncation=True,
        max_length=64
    )

    q9_input_ids.append(encoded["input_ids"])
    q9_attention_masks.append(encoded["attention_mask"])
    q9_labels.append(int(row["label"]))


# Create Hugging Face Dataset
train_dataset_q9 = Dataset.from_dict({
    "input_ids": q9_input_ids,
    "attention_mask": q9_attention_masks,
    "labels": q9_labels
})


print("Training Examples :", len(train_dataset_q9))
print(
    "First Input Shape:",
    torch.tensor(train_dataset_q9[0]["input_ids"]).shape
)


# ----------------------------------------------------------
# Step 2: Load a fresh BERT multiple-choice model
# ----------------------------------------------------------

base_model_q9 = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)


# ----------------------------------------------------------
# Step 3: Configure and apply LoRA
# ----------------------------------------------------------

lora_config_q9 = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

fine_tuned_model = get_peft_model(
    base_model_q9,
    lora_config_q9
)


# ----------------------------------------------------------
# Step 4: Define training arguments
# ----------------------------------------------------------

training_args = TrainingArguments(
    output_dir="./q9_lora_output",

    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,

    max_steps=4,

    logging_steps=1,
    save_strategy="no",
    report_to="none",

    remove_unused_columns=False
)


# ----------------------------------------------------------
# Step 5: Create Hugging Face Trainer
# ----------------------------------------------------------

trainer = Trainer(
    model=fine_tuned_model,
    args=training_args,
    train_dataset=train_dataset_q9
)


# ----------------------------------------------------------
# Step 6: Run tiny LoRA fine-tuning
# ----------------------------------------------------------

train_result = trainer.train()


# ----------------------------------------------------------
# Step 7: Inspect final global step
# ----------------------------------------------------------

print("\nFinal Global Step :", trainer.state.global_step)

Training Examples : 32
First Input Shape: torch.Size([5, 64])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3177.06it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Cons

Step,Training Loss
1,1.719014
2,1.637275
3,1.666976
4,1.573955



Final Global Step : 4


### Conclusion

The final `global_step` reported by the Hugging Face Trainer is **4**.

This matches the specified `max_steps=4` configuration, which limits the fine-tuning process to exactly four optimizer update steps.

# Question 10: Probability Assigned to Option E After Fine-Tuning

## Objective

Use the LoRA fine-tuned multiple-choice model from Question 9 to perform inference on row index **0**.

Apply the Softmax function to the five output logits to convert them into probabilities for answer options A, B, C, D, and E.

Finally, extract the probability assigned to **Option E** and round it to four decimal places.

In [11]:
# ==========================================================
# Question 10
# Probability Assigned to Option E After LoRA Fine-Tuning
# ==========================================================

import torch

# Select row index 0
row_0 = train.iloc[0]

prompt = str(row_0["prompt"])

option_letters = ["A", "B", "C", "D", "E"]

# Create the five prompt-option pairs
formatted_inputs = [
    prompt + " [SEP] " + str(row_0[option])
    for option in option_letters
]

# Tokenize using the same max_length as Q9
encoded_q10 = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

# Add batch dimension:
# [5, 64] -> [1, 5, 64]
input_ids_q10 = encoded_q10["input_ids"].unsqueeze(0)
attention_mask_q10 = encoded_q10["attention_mask"].unsqueeze(0)

# Put model in evaluation mode
fine_tuned_model.eval()

# Run inference without gradient calculation
with torch.no_grad():
    outputs_q10 = fine_tuned_model(
        input_ids=input_ids_q10,
        attention_mask=attention_mask_q10
    )

# Extract five logits
logits_q10 = outputs_q10.logits

# Convert logits into probabilities
probabilities_q10 = torch.softmax(logits_q10, dim=-1)

# Option E corresponds to index 4
option_e_probability = probabilities_q10[0, 4].item()

print("Logits        :", logits_q10)
print("Probabilities :", probabilities_q10)
print("Probability Sum :", probabilities_q10.sum().item())
print("Option E Probability :", option_e_probability)
print("Rounded Answer       :", round(option_e_probability, 4))

Logits        : tensor([[-0.1773, -0.2007, -0.1938, -0.1970, -0.1952]])
Probabilities : tensor([[0.2031, 0.1984, 0.1998, 0.1992, 0.1995]])
Probability Sum : 0.9999999403953552
Option E Probability : 0.19951339066028595
Rounded Answer       : 0.1995


### Conclusion

The probability assigned to **Option E** after tiny LoRA fine-tuning is **0.1995**.

The Softmax function converted the model's five raw logits into a probability distribution across answer options A-E, with the total probability summing to approximately 1.